# MoCo — Momentum Contrast for Unsupervised Visual Representation Learning (2019)

_Papers · Self-supervised & Representation_

**Learn image features with no labels by matching two views of the same image against a big queue of negatives, kept consistent by a slowly-moving key encoder.**

---

This notebook is a practice scaffold for the **MoCo — Momentum Contrast for Unsupervised Visual Representation Learning (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the lesson's two worked examples. ---
# (a) Momentum update (Eqn. 2): theta_k <- m*theta_k + (1-m)*theta_q,  m=0.999
m = 0.999
tk = torch.tensor([0.40, -0.20]); tq = torch.tensor([0.60, 0.10])
print("worked momentum:", [round(v, 5) for v in (m*tk + (1-m)*tq).tolist()])
# worked momentum: [0.4002, -0.1997]

# (b) InfoNCE (Eqn. 1): 1 positive + K=2 negatives, tau=0.07, sims [0.8, 0.5, 0.3]
tau0 = 0.07
sims = torch.tensor([0.8, 0.5, 0.3])
loss0 = F.cross_entropy((sims/tau0).unsqueeze(0), torch.tensor([0]))   # positive in slot 0
print("worked InfoNCE loss:", round(loss0.item(), 4))           # ~0.0144


# --- 1. Unlabeled toy data: 8 clusters in 16-D. An "instance" = center + identity offset. ---
Din, Dz, Kcls = 16, 8, 8
gC = torch.Generator().manual_seed(7)
centers = torch.randn(Kcls, Din, generator=gC) * 1.2
def gen(nb, gen):
    y = torch.randint(0, Kcls, (nb,), generator=gen)
    x = centers[y] + torch.randn(nb, Din, generator=gen) * 0.7   # per-instance identity
    return x, y
def aug(x):                                                       # two views = two noise draws
    return x + torch.randn_like(x) * 0.5

def make_enc():
    torch.manual_seed(3)                                         # SAME init for fq, fk, and the baseline
    return nn.Sequential(nn.Linear(Din, 64), nn.ReLU(), nn.Linear(64, Dz)).to(device)


# --- 2. MoCo training loop. mm<1.0 -> momentum update; mm=1.0 -> ABLATION (key never updates). ---
def train_moco(mm, steps=500, lr=0.05, tau=0.2, Q=512):
    fq = make_enc(); fk = make_enc()
    fk.load_state_dict(fq.state_dict())                          # identical start (Algorithm 1)
    for p in fk.parameters():
        p.requires_grad = False                                 # key encoder gets NO gradients
    queue = F.normalize(torch.randn(Q, Dz, device=device), dim=1)
    ptr = 0
    opt = torch.optim.SGD(fq.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    g = torch.Generator().manual_seed(42); losses = []
    for s in range(steps):
        x, _ = gen(128, g)
        xq, xk = aug(x).to(device), aug(x).to(device)           # two augmented views
        q = F.normalize(fq(xq), dim=1)                          # query
        with torch.no_grad():
            k = F.normalize(fk(xk), dim=1)                      # positive key (no grad)
        l_pos = (q * k).sum(1, keepdim=True)                    # N x 1   : q . k+
        l_neg = q @ queue.t()                                   # N x K   : q . negatives
        logits = torch.cat([l_pos, l_neg], dim=1) / tau         # Eqn. 1 numerator/denominator scores
        labels = torch.zeros(q.size(0), dtype=torch.long, device=device)   # positive is column 0
        loss = F.cross_entropy(logits, labels)                  # InfoNCE
        opt.zero_grad(); loss.backward(); opt.step()            # update fq only
        losses.append(loss.item())
        with torch.no_grad():
            if mm < 1.0:                                         # Eqn. 2: momentum update of fk
                for pk, pq in zip(fk.parameters(), fq.parameters()):
                    pk.data = mm * pk.data + (1 - mm) * pq.data
            bs = k.size(0)                                       # enqueue new keys, dequeue oldest
            queue[ptr:ptr+bs] = k.detach() if ptr+bs <= Q else k.detach()[:Q-ptr]
            ptr = (ptr + bs) % Q
    return fq, losses


# --- 3. Linear probe: freeze features, fit ONE linear layer, report test accuracy. ---
gtr = torch.Generator().manual_seed(11); gte = torch.Generator().manual_seed(99)
xtr, ytr = gen(800, gtr); xte, yte = gen(800, gte)
xtr, xte, ytr, yte = xtr.to(device), xte.to(device), ytr.to(device), yte.to(device)

def probe(enc):
    with torch.no_grad():
        ftr = F.normalize(enc(xtr), dim=1); fte = F.normalize(enc(xte), dim=1)
    clf = nn.Linear(Dz, Kcls).to(device)
    o = torch.optim.Adam(clf.parameters(), lr=0.05)
    for _ in range(500):
        o.zero_grad(); F.cross_entropy(clf(ftr), ytr).backward(); o.step()
    return (clf(fte).argmax(1) == yte).float().mean().item()

print("\ntraining MoCo (m=0.99) ...")
fq_moco, loss_moco = train_moco(0.99)
print("training ABLATION (m=1.0, key encoder frozen) ...")
fq_abl,  loss_abl  = train_moco(1.0)
acc_moco = probe(fq_moco)
acc_abl  = probe(fq_abl)
acc_rand = probe(make_enc())                                    # random untrained encoder baseline

print("\nlinear-probe accuracy")
print("  MoCo (m=0.99) features : %.3f" % acc_moco)
print("  ABLATION (m=1.0)       : %.3f" % acc_abl)
print("  random encoder         : %.3f" % acc_rand)
print("final InfoNCE loss  MoCo %.3f  vs  ablation %.3f" % (loss_moco[-1], loss_abl[-1]))
# MoCo features beat the random encoder; the m=1.0 ablation (stale key encoder) lands in between
# with a higher, non-decreasing InfoNCE loss.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does MoCo (momentum encoder + queue) learn useful unlabeled features, and does the momentum update matter? Linear-probe accuracy + InfoNCE loss vs an m=1.0 ablation._

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# Reproduces the qualitative MoCo effect on unlabeled toy data: a linear probe on the
# learned features beats a random encoder, and the momentum update (Eqn. 2) is what
# keeps the InfoNCE loss falling. (See the CODE block above for the full annotated loop.)
torch.manual_seed(0)
Din, Dz, Kcls = 16, 8, 8
gC = torch.Generator().manual_seed(7)
centers = torch.randn(Kcls, Din, generator=gC) * 1.2
def gen(nb, gen):
    y = torch.randint(0, Kcls, (nb,), generator=gen)
    return centers[y] + torch.randn(nb, Din, generator=gen) * 0.7, y
def aug(x): return x + torch.randn_like(x) * 0.5
def make_enc():
    torch.manual_seed(3)
    return nn.Sequential(nn.Linear(Din, 64), nn.ReLU(), nn.Linear(64, Dz))

def train_moco(mm, steps=500, tau=0.2, Q=512):
    fq, fk = make_enc(), make_enc(); fk.load_state_dict(fq.state_dict())
    for p in fk.parameters(): p.requires_grad = False
    queue = F.normalize(torch.randn(Q, Dz), dim=1); ptr = 0
    opt = torch.optim.SGD(fq.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-4)
    g = torch.Generator().manual_seed(42); losses = []
    for s in range(steps):
        x, _ = gen(128, g); q = F.normalize(fq(aug(x)), dim=1)
        with torch.no_grad(): k = F.normalize(fk(aug(x)), dim=1)
        logits = torch.cat([(q*k).sum(1, keepdim=True), q @ queue.t()], 1) / tau
        loss = F.cross_entropy(logits, torch.zeros(q.size(0), dtype=torch.long))
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        with torch.no_grad():
            if mm < 1.0:
                for pk, pq in zip(fk.parameters(), fq.parameters()):
                    pk.data = mm*pk.data + (1-mm)*pq.data            # Eqn. 2
            bs = k.size(0)
            queue[ptr:ptr+bs] = k.detach() if ptr+bs <= Q else k.detach()[:Q-ptr]
            ptr = (ptr + bs) % Q
    return fq, losses

gtr = torch.Generator().manual_seed(11); gte = torch.Generator().manual_seed(99)
xtr, ytr = gen(800, gtr); xte, yte = gen(800, gte)
def probe(enc):
    with torch.no_grad():
        ftr, fte = F.normalize(enc(xtr), dim=1), F.normalize(enc(xte), dim=1)
    clf = nn.Linear(Dz, Kcls); o = torch.optim.Adam(clf.parameters(), lr=0.05)
    for _ in range(500):
        o.zero_grad(); F.cross_entropy(clf(ftr), ytr).backward(); o.step()
    return (clf(fte).argmax(1) == yte).float().mean().item()

fq_moco, lm = train_moco(0.99); fq_abl, la = train_moco(1.0)
print("probe  MoCo %.3f  ablation %.3f  random %.3f"
      % (probe(fq_moco), probe(fq_abl), probe(make_enc())))
# probe  MoCo 0.993  ablation 0.933  random 0.808   (our run, not the paper's numbers)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working MoCo whose linear probe beats a random encoder. Set the
            momentum $m=1.0$ so the key encoder never updates (it stays at its initial random weights),
            retrain, and re-probe. What happens to the InfoNCE training loss and to the probe accuracy, and
            what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the momentum: skip the Eqn. 2 update so $\theta_k$ is frozen at init; keep encoders, queue, $\tau$, optimizer, and data identical. — _An honest ablation changes exactly one thing &mdash; the momentum update &mdash; so any difference is attributable to it._
- Retrain and watch the InfoNCE loss: it stays high / does not fall the way the $m=0.99$ run's did, because $f_q$ is chasing a fixed, mismatched target encoder. — _If $f_k$ never tracks $f_q$, the positive key is produced by an encoder that disagrees with the query encoder, so the positive can't be made reliably most-similar &mdash; the loss can't drop much._
- Linear-probe the frozen $f_q$: accuracy is noticeably lower than the $m=0.99$ run. — _A worse contrastive objective yields worse features, which a linear probe exposes directly._

**Answer:** With $m=1.0$ the key encoder is frozen at random init, so $f_q$ chases a fixed, inconsistent
                 target: the InfoNCE loss stays high (in our run ~4.4 vs ~3.6 for $m=0.99$) and the linear-probe
                 accuracy drops (~0.93 vs ~0.99 in our run). Since the only change is the momentum update, this
                 isolates Eqn. 2 as the mechanism that keeps the dictionary consistent enough to learn good
                 features &mdash; matching the paper's momentum ablation (&sect;4.1). The CODEVIZ panel shows the
                 two loss curves.

</details>

**Problem 2.** Recompute the momentum update by hand. The key weight is $\theta_k = [1.0,\,0.0]$, the query weight
            is $\theta_q = [0.0,\,1.0]$, and $m=0.99$. What is the new $\theta_k$, and how far did it move
            toward $\theta_q$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply Eqn. 2 component-wise: $\theta_k \leftarrow 0.99\,\theta_k + 0.01\,\theta_q$. — _The update is an element-wise exponential moving average._
- Component 0: $0.99\times 1.0 + 0.01\times 0.0 = 0.99$. Component 1: $0.99\times 0.0 + 0.01\times 1.0 = 0.01$. — _Keep $99\%$ of the old weight, add $1\%$ of the query weight._
- Compare to the $m=0.999$ case: there it would be $[0.999,\,0.001]$ &mdash; an even smaller step. — _Larger $m$ &rarr; slower key encoder &rarr; more-consistent dictionary._

**Answer:** $\theta_k \leftarrow [0.99,\,0.01]$ &mdash; it moved just $1\%$ of the way toward
                 $\theta_q$. With $m=0.999$ it would move only $0.1\%$ (to $[0.999,0.001]$). The bigger the
                 momentum, the slower and more consistent the key encoder, which is why the paper defaults to
                 $m=0.999$.

</details>

**Problem 3.** Your InfoNCE worked example had positive similarity $0.8$ and negatives $0.5, 0.3$ ($\tau=0.07$),
            giving loss $\approx 0.0144$. Now suppose one negative also scores $0.8$ (a "hard negative"): the
            similarities are positive $0.8$, negatives $0.8$ and $0.3$. Does the loss go up or down, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Logits $= [0.8,0.8,0.3]/0.07 = [11.4286, 11.4286, 4.2857]$; exponentiate: $[91910.6, 91910.6, 72.7]$, sum $= 183893.9$. — _The hard negative now has the same score as the positive, so its $\exp$ term equals the positive's._
- Loss $= -\log(91910.6 / 183893.9) = -\log(0.49980) \approx 0.6933$. — _The positive now holds only ~half the softmax mass instead of ~99%._
- Compare: $0.6933 \gg 0.0144$ &mdash; the loss jumped by ~48×. — _InfoNCE punishes a negative that looks as similar as the positive; that is exactly the signal that trains the encoder._

**Answer:** The loss rises sharply, from $\approx 0.0144$ to $\approx 0.693$. A negative scoring as
                 high as the positive splits the softmax mass roughly in half, so $-\log(0.4998)\approx0.693$.
                 This is the point of contrastive learning: hard negatives (other images that currently
                 look like the query) produce the large gradients that push the encoder to separate them &mdash;
                 and a large queue makes such hard negatives more likely to be present.

</details>